# TenderWin · Лід і мейл генератор v1.4 — ТЕСТОВИЙ РЕЖИМ

Вісім клітинок, згори вниз. Код редагувати не треба.

**Що робить:** шукає в Prozorro відхилення за поточну добу по всіх галузях і
всіх закупівлях, складає перший лист (V9) і робочу картку закупівлі, а для
кожного відхилення — **службову картку і протокол (рішення) замовника** у
`Мій диск / 0 Cases / <ID закупівлі> <ЄДРПОУ або ІПН учасника>`
(на компʼютері це `G:\Мій диск\0 Cases`).

**Чого не робить:** не завантажує документи пропозиції й тендерну
документацію, не аналізує їх, не пише скарг і **не надсилає нічого
клієнтам** — тільки на ваші тестові скриньки.

Потрібні чотири файли: `tenderwin_lead_mail.py`, `tenderwin_lead_engine.py`,
`lead_machine_v1.py`, `tenderwin_service_cards.py`. Якщо їх ще немає на
Диску — клітинка 2 запропонує завантажити їх із компʼютера.

## КЛІТИНКА 1 · Встановлення бібліотек

In [ ]:
!pip -q install google-api-python-client google-auth google-auth-oauthlib python-docx requests pdfplumber pypdf openpyxl
print("Готово. Далі — клітинка 2.")

## КЛІТИНКА 2 · Google Диск, робоча тека і файли

Монтує Диск і працює з текою **`1 Lead&Mail generator`** (назва — у рядку
`РОБОЧА_ТЕКА`). Там мають лежати чотири файли: `tenderwin_lead_mail.py`,
`tenderwin_lead_engine.py`, `lead_machine_v1.py`, `tenderwin_service_cards.py`.
Якщо якогось бракує — відкриється вікно вибору файлів на компʼютері.

**Перший запуск у новій теці:** база `tenderwin_lead_mail.db` (кому вже
писали), `leads.db` і токен `.gmail_token.json` один раз копіюються сюди з
попередньої теки `TENDERWIN`. Там нічого не видаляється і не змінюється, а
вже наявна база в робочій теці ніколи не перезаписується.

In [ ]:
from google.colab import drive
import os, re, shutil, sqlite3, sys, time

drive.mount('/content/drive')

КОРІНЬ = '/content/drive/MyDrive'
#: Робоча тека: скрипти, ноутбук, база, листи. Змінювати назву — лише тут.
РОБОЧА_ТЕКА = '1 Lead&Mail generator'
#: Де база й токен лежали раніше. Звідти їх ОДИН раз копіюють у робочу теку.
ПОПЕРЕДНІ_ТЕКИ = ('tenderwin',)          # будь-який регістр: TENDERWIN, TenderWin…
ПОТРІБНІ = ['tenderwin_lead_mail.py', 'tenderwin_lead_engine.py',
            'lead_machine_v1.py', 'tenderwin_service_cards.py']
#: Стан, який переїжджає разом із запуском. Без бази скрипт «забув» би,
#: кому вже писав перший лист, і написав би ще раз.
СТАН = ['tenderwin_lead_mail.db', 'leads.db', '.gmail_token.json']
БАЗА = 'tenderwin_lead_mail.db'


def вміст(тека):
    try:
        return set(os.listdir(тека))
    except OSError:
        return set()


def база_порожня(шлях):
    """True — у базі ще немає жодної компанії, події, листа чи стопу."""
    try:
        with sqlite3.connect(шлях) as зʼєднання:
            рядків = sum(зʼєднання.execute(f"SELECT COUNT(*) FROM {т}").fetchone()[0]
                         for т in ('companies', 'events', 'outreach', 'suppressions'))
        return рядків == 0
    except sqlite3.Error:
        return False          # не прочитали — вважаємо цінною і не чіпаємо


def скопіювати(звідки, куди):
    # Лише вміст: права й дати файлів Google Диск у Colab може не прийняти.
    shutil.copyfile(звідки, куди)
    # Журнал SQLite поруч із базою — частина бази, якщо він не порожній.
    for хвіст in ('-wal', '-journal'):
        if os.path.exists(звідки + хвіст) and os.path.getsize(звідки + хвіст) > 0:
            shutil.copyfile(звідки + хвіст, куди + хвіст)


DYSK = os.path.join(КОРІНЬ, РОБОЧА_ТЕКА)
if not os.path.isdir(DYSK):
    os.makedirs(DYSK, exist_ok=True)
    print("Теки не було — створив:", DYSK)
print("Робоча тека:", DYSK)

# --- База, історія і токен: один раз із попередньої теки -------------------
попередні = [os.path.join(КОРІНЬ, назва) for назва in sorted(os.listdir(КОРІНЬ))
             if назва.lower() in ПОПЕРЕДНІ_ТЕКИ
             and os.path.isdir(os.path.join(КОРІНЬ, назва))
             and os.path.abspath(os.path.join(КОРІНЬ, назва)) != os.path.abspath(DYSK)]
# Справжня попередня тека — та, де найбільша база.
попередні.sort(key=lambda т: os.path.getsize(os.path.join(т, БАЗА))
               if os.path.exists(os.path.join(т, БАЗА)) else -1, reverse=True)
ДЖЕРЕЛО = next((т for т in попередні
                if any(os.path.exists(os.path.join(т, н)) for н in СТАН)), None)

if ДЖЕРЕЛО:
    перенесено = []
    for назва in СТАН:
        звідки, куди = os.path.join(ДЖЕРЕЛО, назва), os.path.join(DYSK, назва)
        if not os.path.exists(звідки):
            continue
        if os.path.exists(куди):
            if назва == БАЗА and база_порожня(куди) and not база_порожня(звідки):
                # Порожню базу міг створити запуск до перенесення — відкладаємо
                # її поруч (не видаляємо) і беремо справжню.
                відкладена = куди + time.strftime('.empty_%Y%m%d-%H%M%S')
                os.replace(куди, відкладена)
                скопіювати(звідки, куди)
                перенесено.append(f"{назва} (порожню відкладено: {os.path.basename(відкладена)})")
            elif назва == БАЗА and os.path.getsize(куди) < os.path.getsize(звідки) \
                    and not база_порожня(куди):
                print(f"УВАГА: у робочій теці вже є {БАЗА}, і вона менша, ніж у {ДЖЕРЕЛО}.")
                print("   Скрипт її НЕ перезаписує. Якщо обидві бази вели листи —")
                print("   історія розійшлась: не відправляйте листи (клітинка 7),")
                print("   доки бази не об'єднано.")
            continue
        скопіювати(звідки, куди)
        перенесено.append(назва)
    if перенесено:
        print(f"Скопійовано з {ДЖЕРЕЛО}: " + ", ".join(перенесено))
        print("   Там усе лишилось як було — це резервна копія. Далі запускайте")
        print("   лише звідси: дві теки з двома базами «забудуть», кому вже писали.")
        if '.gmail_token.json' in перенесено:
            print("   ⓘ Токен пошти — тепер і в цій теці. Надійніше тримати його")
            print("     в Colab Secrets (GMAIL_OAUTH_JSON), а файл видалити.")

# --- Модулі лише з робочої теки ---------------------------------------------
# Якщо в цьому сеансі вже працювали з іншою текою, Python пам'ятає її модулі
# і reload перечитав би старі файли. Забуваємо їх і ставимо робочу теку першою.
for назва_модуля in [н[:-3] for н in ПОТРІБНІ]:
    модуль = sys.modules.get(назва_модуля)
    файл = getattr(модуль, '__file__', '') or ''
    if модуль is not None and os.path.dirname(os.path.abspath(файл)) != os.path.abspath(DYSK):
        del sys.modules[назва_модуля]
sys.path[:] = [шлях for шлях in sys.path
               if not os.path.exists(os.path.join(шлях or '.', 'tenderwin_lead_mail.py'))]
sys.path.insert(0, DYSK)


def чого_бракує():
    файли = вміст(DYSK)
    return [назва for назва in ПОТРІБНІ if назва not in файли]


def справжня_назва(назва):
    # Браузер часто додає "(1)": tenderwin_lead_mail (1).py
    чиста = re.sub(r"\s*\(\d+\)(?=\.py$)", "", назва).strip()
    return чиста if чиста in ПОТРІБНІ else назва


бракує = чого_бракує()
ГОТОВО = not бракує

if ГОТОВО:
    print("Усі файли на місці. Далі — клітинка 3.")
else:
    print("Бракує:", ", ".join(бракує))
    print("Оберіть ці файли на компʼютері у вікні нижче.")
    print("(Або покладіть їх у цю теку на Диску і виконайте клітинку ще раз.)")
    try:
        from google.colab import files
        завантажені = files.upload()
        for тимчасова in list(завантажені):
            назва = справжня_назва(тимчасова)
            if назва in ПОТРІБНІ:
                куди = os.path.join(DYSK, назва)
                if os.path.exists(куди):
                    os.remove(куди)
                shutil.move(тимчасова, куди)
                print("  покладено на Диск:", назва)
            else:
                print("  пропущено (не потрібен тут):", тимчасова)
    except Exception as помилка:
        print("Вікно вибору не відкрилося:", type(помилка).__name__)

    бракує = чого_бракує()
    ГОТОВО = not бракує
    print()
    print("Усі файли на місці. Далі — клітинка 3." if ГОТОВО
          else "Досі бракує: " + ", ".join(бракує) + ". Виконайте клітинку ще раз.")

## КЛІТИНКА 3 · Доступ до пошти

Секрет `GMAIL_OAUTH_JSON` має бути в Colab Secrets (ключик ліворуч), доступ
для цього ноутбука — увімкнений. У код секрети не вставляються ніколи.

In [ ]:
import sys, os

if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2 — вона монтує Диск.")
    ФАЙЛИ_Є = False
else:
    if DYSK not in sys.path:
        sys.path.insert(0, DYSK)
    try:
        import tenderwin_lead_engine as E
        ФАЙЛИ_Є = True
    except ModuleNotFoundError as помилка:
        ФАЙЛИ_Є = False
        print("Немає файла:", (помилка.name or "?") + ".py")
        print("Поверніться до клітинки 2 — вона покладе файли на Диск.")

if ФАЙЛИ_Є:
    # Без цього рядка токен, що лежить файлом на Диску, просто не видно:
    # движок шукає .gmail_token.json у теці, яку йому назвали.
    E.TOKEN_DIR = DYSK
    у_теці = os.path.exists(os.path.join(DYSK, E.TOKEN_FILE_NAME))

    if E._get_secret(E.GMAIL_OAUTH_ENV):
        print("Доступ до пошти є" + (f" (файл {E.TOKEN_FILE_NAME} у теці на Диску)."
                                     if у_теці else " (секрет Colab)."))
        бракує_дозволів = E.missing_scopes()
        if бракує_дозволів:
            print("  Але токену бракує дозволів:", ", ".join(бракує_дозволів))
            print("  Зробіть E.reset_gmail_token(DYSK), потім E.mint_oauth_token().")
        else:
            print("Далі — клітинка 4.")
        if у_теці:
            print("  ⓘ Токен у теці на Диску може прочитати кожен, хто має")
            print("    доступ до цієї теки. Надійніше тримати його в Colab Secrets.")
    else:
        print(f"Доступу до пошти немає: ні секрету {E.GMAIL_OAUTH_ENV},")
        print(f"ні файла {E.TOKEN_FILE_NAME} у теці {DYSK}.")
        print("Без нього працює сухий прогін (клітинка 6), відправки не буде.")
        print("Як отримати токен: E.mint_oauth_token()")

## КЛІТИНКА 4 · Завантаження скрипта

Перевіряє, що на Диску саме версія 1.4.0. Якщо там старий файл —
зупиняється і пише, який саме файл замінити.

In [ ]:
import importlib, os

# Версії, з якими працює цей ноутбук. Старіший файл на Диску — клітинка
# зупиниться й назве його, а не впаде далі з незрозумілою помилкою.
ПОТРІБНІ_ВЕРСІЇ = {"tenderwin_lead_mail": "1.4.0", "tenderwin_service_cards": "1.1.0"}


def номер(версія):
    try:
        return tuple(int(частина) for частина in str(версія).split("."))
    except ValueError:
        return (0,)


M = None
if "DYSK" not in globals():
    print("Спершу виконайте клітинку 2.")
else:
    модулі = {}
    for назва in ПОТРІБНІ_ВЕРСІЇ:
        try:
            модуль = importlib.import_module(назва)
            модулі[назва] = importlib.reload(модуль)
        except ModuleNotFoundError as помилка:
            print("Немає файла:", (помилка.name or назва) + ".py")
            print("Поверніться до клітинки 2.")
            модулі = None
            break
    if модулі:
        застарілі = []
        for назва, треба in ПОТРІБНІ_ВЕРСІЇ.items():
            модуль = модулі[назва]
            є = getattr(модуль, "VERSION", "?")
            if номер(є) < номер(треба):
                шлях = getattr(модуль, "__file__", "") or ""
                розмір = os.path.getsize(шлях) if шлях and os.path.exists(шлях) else 0
                застарілі.append((назва, є, треба, шлях, розмір))
        if застарілі:
            print("СТОП: на Диску стара версія скрипта.")
            for назва, є, треба, шлях, розмір in застарілі:
                print(f"   {назва}.py — версія {є}, а потрібна {треба}")
                print(f"      файл: {шлях} ({розмір:,} байтів)".replace(",", " "))
            print()
            print("Що зробити:")
            print(f"  1. Покладіть новий файл із пакета 1.4.0 у теку {DYSK}")
            print("     (у вікні Google Диска оберіть «Замінити наявний файл»,")
            print("     а не «Зберегти обидва»).")
            print("  2. Меню «Середовище виконання» → «Перезапустити сеанс».")
            print("  3. Клітинки 2, 3, 4 ще раз.")
        else:
            M = модулі["tenderwin_lead_mail"]
            print(f"tenderwin_lead_mail {M.VERSION} · "
                  f"tenderwin_service_cards {модулі['tenderwin_service_cards'].VERSION}"
                  f" · з теки {os.path.dirname(M.__file__)}")

if M is not None:
    M.setup(DYSK)      # база, білий список адрес, історія попереднього движка

## КЛІТИНКА 5 · Налаштування

Змінювати не обовʼязково. Значення нижче — чинне рішення власника.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.LETTER_VERSION = "V9_OWNER"       # ЧИННИЙ лист (рішення 06.10.2026)
    M.LETTER_FORMAT  = "HTML_AND_TEXT"  # "TEXT_ONLY" — лист лише текстом
    M.WINDOW_MODE    = "TODAY_ONLY"     # лише поточна доба
    M.SKIP_WITHOUT_COMPLAINT_ROUTE = True   # без строку оскарження — не пишемо
    M.MIN_LOT_VALUE  = 0                # 0 = без порогу вартості
    M.SERVICE_CARDS  = True             # службові картки з протоколами
    M.SERVICE_CARDS_FRESH = True        # свіжий стан рішення з Prozorro
    M.CASES_DIR = ""                    # "" = «Мій диск/0 Cases» (G:\Мій диск\0 Cases)

    print("режим:", M.MODE, "· лист:", M.LETTER_VERSION, "·", M.LETTER_FORMAT)
    print("адрес для тестової доставки:", len(M.TEST_RECIPIENTS))
    print("службові картки:", "так" if M.SERVICE_CARDS else "ні")
    if M.SERVICE_CARDS:
        print("справи (картка + протокол):", M.cases_root())

## КЛІТИНКА 6 · Сухий прогін

Шукає відхилення за сьогодні, складає листи у файли і робочі картки.
**Нічого нікуди не надсилає.**

У теці `vyhid_lystiv/eml` біля кожного листа лежить файл `.html` — відкрийте
його у браузері, щоб побачити лист таким, яким його побачить клієнт.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    результат = M.go(dysk=DYSK)

## КЛІТИНКА 7 · Відправка на ваші скриньки і стан бази

Листи підуть **тільки** на адреси зі списку `TEST_RECIPIENTS`. База це
перевіряє окремо від коду: адресу клієнта туди не пропустить тригер.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.send(dysk=DYSK)     # відправка на тестові скриньки + звірка з Gmail
    M.state(dysk=DYSK)    # що зараз у базі

## КЛІТИНКА 8 · Службові картки ще раз (за потреби)

Після `M.go()` картки вже зроблені: кожна справа — окрема тека в
`Мій диск / 0 Cases` з назвою «ID закупівлі + ЄДРПОУ (ІПН) учасника», у ній
службова картка і протокол (рішення) замовника. Ця клітинка потрібна лише
тоді, коли крок карток обірвався або коли треба зібрати картки за інший день.

Уже завантажені файли вдруге не качаються. Картку, у якій ви щось дописали,
скрипт **не перезаписує** — нова версія ляже поруч окремим файлом.

In [ ]:
if globals().get("M") is None:
    print("Спершу виконайте клітинки 2–4.")
else:
    M.kartky(dysk=DYSK)                      # події останнього прогону
    # M.kartky(day="2026-10-02", dysk=DYSK)  # усі події конкретного дня

---
### Якщо щось пішло не так

| Повідомлення | Що робити |
|---|---|
| `Немає файла: …py` | клітинка 2 — покладіть файли на Диск |
| `СТОП: на Диску стара версія скрипта` | замініть названий файл у робочій теці новим із пакета 1.4.0, перезапустіть сеанс, клітинки 2–4 |
| `module 'tenderwin_lead_mail' has no attribute …` | те саме: Colab бачить старий файл — див. рядок вище |
| `УВАГА: у робочій теці вже є tenderwin_lead_mail.db…` | у робочій теці інша, менша база: листи не відправляти, доки бази не об'єднано |
| `TRANSPORT_UNAVAILABLE` | немає секрету `GMAIL_OAUTH_JSON` — клітинка 3 |
| `TEST_ALLOWLIST_EMPTY` | порожній `TEST_RECIPIENTS` — так задумано, заповніть |
| `PROZORRO_UNAVAILABLE` | Prozorro не відповідає — спробуйте за кілька хвилин |
| `Обхід стрічки: INCOMPLETE` | прочитано не всю стрічку; у звіті написано чому |
| `листи без певного стану` | зробіть `M.send(dysk=DYSK)` — він звірить їх із Gmail |
| `SERVICE_CARDS_MODULE_MISSING` | немає `tenderwin_service_cards.py` — клітинка 2; листи цим не зачеплені |
| `SERVICE_CARDS_FAILED` | крок карток обірвався; листи збережено. Повторіть `M.kartky(dysk=DYSK)` |

**Де службові картки і протоколи:** `Мій диск → 0 Cases → <UA-ID> <ЄДРПОУ/ІПН>`
(на компʼютері `G:\Мій диск\0 Cases`). Якщо того самого учасника в тій самій
закупівлі відхилили двічі (два лоти, повторне рішення), друга справа — у теці
з « (2)» в кінці.

**Перелік для відбору:** `1 Lead&Mail generator → Службові картки → <дата> →
_ПЕРЕЛІК_<прогін>.xlsx` — один рядок на подію, колонка «Тека справи».

Перевірити сам скрипт: `M.tests(DYSK)`